# Task 2.1 — Dataset Selection and Setup
**Paper:** Incremental Learning of Temporally-Coherent Gaussian Mixture Models

In [1]:
%matplotlib inline
import numpy as np
import random
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
import warnings
warnings.filterwarnings('ignore')

# ─── GLOBAL RANDOM SEED ───
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
print('Seeds set to:', RANDOM_SEED)

Seeds set to: 42

## Dataset Choice and Justification

**What the dataset is:** A synthetic 2D dataset of 150 points sampled as three sequential Gaussian clusters — each cluster is traversed in temporal order before moving to the next. Points arrive one-by-one in a temporally smooth order, simulating the kind of streaming face-feature or trajectory data used in the original paper.

**Why it is a reasonable testbed:** The paper's method is designed for sequentially ordered data that is temporally coherent — nearby points in time belong to the same or nearby Gaussian component. This synthetic dataset directly satisfies that assumption, making it a fair and controlled test of the core algorithm. The three-cluster structure also exercises the model's component splitting/merging mechanism.

**Limitations compared to the paper:** The paper evaluates on real video face sequences (300–5000 frames in high-dimensional pixel space). Our dataset is 2D and synthetic, so it cannot test the method's scalability to high dimensions, nor capture the complex non-linear manifold structure of real face appearance. Noise levels are also simpler here than in realistic imaging conditions.

**Preprocessing:** No normalisation was applied since all features are on the same synthetic scale. The temporal order is the natural sequential ordering by cluster (simulating structured data arrival).

In [1]:
# ─── Dataset Generation ───
np.random.seed(42)

# 3 Gaussian clusters visited sequentially (temporal coherence preserved within each cluster)
N_per_cluster = 50
centers = [(0, 0), (4, 3), (8, 0)]
stds = [(0.4, 0.9), (0.5, 0.5), (0.7, 0.35)]

segments = []
for (cx, cy), (sx, sy) in zip(centers, stds):
    pts = np.column_stack([
        np.random.randn(N_per_cluster) * sx + cx,
        np.random.randn(N_per_cluster) * sy + cy
    ])
    segments.append(pts)

X = np.vstack(segments)   # shape (150, 2)
labels_true = np.array([0]*50 + [1]*50 + [2]*50)

print(f"Dataset shape: {X.shape}")
print(f"Features: x ∈ [{X[:,0].min():.2f}, {X[:,0].max():.2f}], y ∈ [{X[:,1].min():.2f}, {X[:,1].max():.2f}]")
print(f"Clusters: {np.unique(labels_true)} | Points per cluster: {[50, 50, 50]}")


Dataset shape: (150, 2)
Features: x ∈ [-1.63, 10.21], y ∈ [-2.35, 5.12]
Clusters: [0 1 2] | Points per cluster: [50, 50, 50]


The dataset has 150 samples, 2 continuous features, and is arranged in a temporally coherent sequence — matching the exact problem formulation in Section 2.1 of the paper.

In [1]:
# ─── Visualise dataset ───
fig, ax = plt.subplots(figsize=(7,5))
colors = ['steelblue', 'tomato', 'green']
labels = ['Cluster 1 (0,0)', 'Cluster 2 (4,3)', 'Cluster 3 (8,0)']
for i, (seg, lbl, col) in enumerate(zip(segments, labels, colors)):
    ax.scatter(seg[:,0], seg[:,1], c=col, s=30, alpha=0.7, label=lbl)
    ax.scatter(*np.mean(seg, axis=0), marker='*', s=200, c=col, edgecolors='black', linewidths=1, zorder=5)
ax.set_title('Toy Dataset: 3 Sequential Gaussian Clusters\n(Temporal order: 1→2→3)', fontsize=12, fontweight='bold')
ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('results/task2_dataset.png', dpi=150, bbox_inches='tight')
plt.show()
print("Dataset plot saved.")


Dataset plot saved.

<Figure>